# Bronze - CFPB API Ingestion

## Imports

In [2]:
import requests

from datetime import date, timedelta
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql.types import StructType, StructField, StringType, BooleanType

StatementMeta(, 697faa06-4958-40e2-be61-d8b625772e5b, 4, Finished, Available, Finished, False)

## Configuration

In [3]:
api_url = "https://www.consumerfinance.gov/data-research/consumer-complaints/search/api/v1/"

api_bronze_table = "bronze.cfpb_complaints_api"

page_size = 100
timeout = 10
publication_lag_days = 2

StatementMeta(, 697faa06-4958-40e2-be61-d8b625772e5b, 5, Finished, Available, Finished, False)

## Date Range

In [4]:
run_date = date.today()
target_date = run_date - timedelta(days=publication_lag_days)

start_date = target_date.isoformat()
end_date = (target_date + timedelta(days=1)).isoformat()

StatementMeta(, 697faa06-4958-40e2-be61-d8b625772e5b, 6, Finished, Available, Finished, False)

## API Request

In [5]:
headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json"
}

records = []
page = 1
search_after = None

while True:
    params = {
        "date_received_min": start_date,
        "date_received_max": end_date,
        "size": page_size,
        "sort": "created_date_desc"
    }

    if search_after is not None:
        params["page"] = page
        params["frm"] = (page - 1) * page_size
        params["search_after"] = search_after

    response = requests.get(
        api_url,
        params=params,
        headers=headers,
        timeout=timeout
    )

    response.raise_for_status()
    response_json = response.json()

    raw_records = response_json["hits"]["hits"]

    records.extend([
        row["_source"]
        for row in raw_records
    ])

    break_points = response_json["_meta"].get("break_points", {})
    next_page = str(page + 1)

    if next_page not in break_points:
        break

    search_after_values = break_points[next_page]
    search_after = f"{search_after_values[0]}_{search_after_values[1]}"

    page += 1

if len(records) == 0:
    raise ValueError("No records returned from CFPB API.")

StatementMeta(, 697faa06-4958-40e2-be61-d8b625772e5b, 7, Finished, Available, Finished, False)

## Prepare Bronze Data

In [6]:
bronze_schema = StructType([
    StructField("company", StringType(), True),
    StructField("company_public_response", StringType(), True),
    StructField("company_response", StringType(), True),
    StructField("complaint_id", StringType(), True),
    StructField("complaint_what_happened", StringType(), True),
    StructField("date_received", StringType(), True),
    StructField("date_sent_to_company", StringType(), True),
    StructField("has_narrative", BooleanType(), True),
    StructField("issue", StringType(), True),
    StructField("product", StringType(), True),
    StructField("state", StringType(), True),
    StructField("sub_issue", StringType(), True),
    StructField("sub_product", StringType(), True),
    StructField("submitted_via", StringType(), True),
    StructField("tags", StringType(), True),
    StructField("timely", StringType(), True),
    StructField("zip_code", StringType(), True)
])

bronze_df = spark.createDataFrame(records, bronze_schema)

bronze_df = (
    bronze_df
    .withColumn("source_system", lit("cfpb"))
    .withColumn("api_start_date", lit(start_date))
    .withColumn("api_end_date", lit(end_date))
    .withColumn("bronze_ingestion_timestamp", current_timestamp())
)

StatementMeta(, 697faa06-4958-40e2-be61-d8b625772e5b, 8, Finished, Available, Finished, False)

## Write Bronze Table

In [7]:
(
    bronze_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(api_bronze_table)
)

StatementMeta(, 697faa06-4958-40e2-be61-d8b625772e5b, 9, Finished, Available, Finished, False)